Task 1: Conceptual Understanding 

1. Difference between "Love" and "love" in NLP
"Love" and "love" are treated as different tokens in case-sensitive systems.
Without normalization → model sees them as two separate words
With lowercasing → both become "love" → improves consistency

2. What happens if stopwords are not removed?
Adds noise
Increases dimensionality
Reduces model performance in many tasks

3. When removing stopwords is harmful

Case 1: Sentiment Analysis

"I am not happy" → removing "not" changes meaning

Case 2: Question Answering

"What is the capital of India?" → removing "what", "is" breaks structure

4. Stemming vs Lemmatization
Feature	 Stemming	 Lemmatization
Method	 Rule-based	 Vocabulary + context
Output	 Not always  real word	Real word
Example	"running" → "run"	"running" → "run"
Example	"better" → "better"	"better" → "good"

TASK 2: Preprocessing Function

In [2]:
import re

def preprocess_text(text):
    if not text or not isinstance(text, str):
        return [], ""
    # Remove URLs
    text = re.sub(r"http\S+|www\S+", "", text)

    # Remove emails
    text = re.sub(r"\S+@\S+", "", text)

    # Remove numbers
    text = re.sub(r"\d+", "", text)

    # Lowercase
    text = text.lower()

    # Remove repeated characters (soooo → so)
    text = re.sub(r'(.)\1{2,}', r'\1', text)

    # Remove special characters (keep alphabets & space)
    text = re.sub(r"[^a-z\s]", "", text)

    # Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    tokens = text.split()

    # Remove short tokens except 'no', 'not'
    tokens = [word for word in tokens if len(word) > 2 or word in ["no", "not"]]

    clean_sentence = " ".join(tokens)

    return tokens, clean_sentence

TASK 3: Stress Testing

In [4]:
test_sentences = [
    "Get 100% FREE access now!!!",
    "I absolutely looooved this product ",
    "Worst service ever... 0/10",
    "Call me at 9876543210",
    "This is THE best course!!!",
    "Visit https://openai.com now!",
    "Nooooo this is baaad!!!",
    "OK OK OK I got it",
    "Win $$$ now!!! Limited offer!!!",
    "I am not happy with this"
]

results = []

for text in test_sentences:
    tokens, clean = preprocess_text(text)
    results.append((text, tokens, clean))

for r in results:
    print("Original:", r[0])
    print("Tokens:", r[1])
    print("Cleaned:", r[2])
    print("-"*50)

Original: Get 100% FREE access now!!!
Tokens: ['get', 'free', 'access', 'now']
Cleaned: get free access now
--------------------------------------------------
Original: I absolutely looooved this product 
Tokens: ['absolutely', 'loved', 'this', 'product']
Cleaned: absolutely loved this product
--------------------------------------------------
Original: Worst service ever... 0/10
Tokens: ['worst', 'service', 'ever']
Cleaned: worst service ever
--------------------------------------------------
Original: Call me at 9876543210
Tokens: ['call']
Cleaned: call
--------------------------------------------------
Original: This is THE best course!!!
Tokens: ['this', 'the', 'best', 'course']
Cleaned: this the best course
--------------------------------------------------
Original: Visit https://openai.com now!
Tokens: ['visit', 'now']
Cleaned: visit now
--------------------------------------------------
Original: Nooooo this is baaad!!!
Tokens: ['no', 'this', 'bad']
Cleaned: no this bad
-------

TASK 4: Token Analytics

In [6]:
def token_analysis(tokens):
    if not tokens:
        return 0, 0, 0

    total = len(tokens)
    unique = len(set(tokens))
    avg_len = sum(len(t) for t in tokens) / total

    return total, unique, round(avg_len, 2)

for text, tokens, clean in results:
    total, unique, avg = token_analysis(tokens)
    print(f"Sentence: {clean}")
    print(f"Total: {total}, Unique: {unique}, Avg Length: {avg}")
    print("-"*40)

Sentence: get free access now
Total: 4, Unique: 4, Avg Length: 4.0
----------------------------------------
Sentence: absolutely loved this product
Total: 4, Unique: 4, Avg Length: 6.5
----------------------------------------
Sentence: worst service ever
Total: 3, Unique: 3, Avg Length: 5.33
----------------------------------------
Sentence: call
Total: 1, Unique: 1, Avg Length: 4.0
----------------------------------------
Sentence: this the best course
Total: 4, Unique: 4, Avg Length: 4.25
----------------------------------------
Sentence: visit now
Total: 2, Unique: 2, Avg Length: 4.0
----------------------------------------
Sentence: no this bad
Total: 3, Unique: 3, Avg Length: 3.0
----------------------------------------
Sentence: got
Total: 1, Unique: 1, Avg Length: 3.0
----------------------------------------
Sentence: win now limited offer
Total: 4, Unique: 4, Avg Length: 4.5
----------------------------------------
Sentence: not happy with this
Total: 4, Unique: 4, Avg Length: 

TASK 5: Frequency Analysis

In [7]:
from collections import Counter

all_tokens = []

for _, tokens, _ in results:
    all_tokens.extend(tokens)

counter = Counter(all_tokens)

print("Top 10 frequent:", counter.most_common(10))
print("Least 5 frequent:", counter.most_common()[-5:])

Top 10 frequent: [('this', 4), ('now', 3), ('get', 1), ('free', 1), ('access', 1), ('absolutely', 1), ('loved', 1), ('product', 1), ('worst', 1), ('service', 1)]
Least 5 frequent: [('limited', 1), ('offer', 1), ('not', 1), ('happy', 1), ('with', 1)]


TASK 6: Full Pipeline

In [9]:
def full_pipeline(text_list):
    all_tokens = []
    clean_sentences = []

    for text in text_list:
        tokens, clean = preprocess_text(text)
        all_tokens.extend(tokens)
        clean_sentences.append(clean)

    return {
        "tokens": all_tokens,
        "clean_sentences": clean_sentences
    }